<a href="https://colab.research.google.com/github/ladparag100/AI-Projects/blob/main/1.%20Smart%20CSV%20AI%20Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-experimental langchain-openai pandas requests==2.32.4
!pip install gdown

In [ ]:
import pandas as pd
import os
import sys
import gdown
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# ==========================================
#  PART 1: AUTOMATIC FILE DOWNLOADER
# ==========================================

files_to_download = {
    "saas_docs.csv":         "https://drive.google.com/file/d/1RElOhN7bYsDAJUNQhYyqM7IzX-Xo6myq/view?usp=sharing",
    "credit_card_terms.csv": "https://drive.google.com/file/d/1_giivc_B0urOKpct0XY2yVZuxW3Eenuf/view?usp=sharing",
    "hospital_policy.csv":   "https://drive.google.com/file/d/1pL7OnDhnmz9pteIpBJ12gu2_ixrc2hPm/view?usp=sharing",
    "ecommerce_faqs.csv":    "https://drive.google.com/file/d/1O4fTjsLFbz55oOiwJUwLwZryO5OSSF6p/view?usp=sharing"
}

print("--- Downloading Files from Google Drive ---")
for filename, url in files_to_download.items():
    if not os.path.exists(filename):
        gdown.download(url, filename, quiet=False, fuzzy=True)
        print(f"Downloaded: {filename}")
    else:
        print(f"Skipped: {filename} (Already exists)")
print("--- Download Complete ---\n")


# ==========================================
#  PART 2: AI AGENT SETUP (MULTI-FILE)
# ==========================================

# 1. SETUP: Get API Key Securely
print("ENTER YOUR OPENAI API KEY BELOW:")
api_key = getpass()

# 2. LOAD ALL CSVs INTO A LIST
dataframes = [] # We will store all the loaded tables here
loaded_names = []

try:
    for filename in files_to_download.keys():
        df = pd.read_csv(filename)
        dataframes.append(df)
        loaded_names.append(filename)
        print(f"SUCCESS: Loaded '{filename}' ({len(df)} rows)")

except Exception as e:
    print(f"\nERROR loading files: {e}")
    sys.exit()

# 3. DEFINE THE RULES
system_prompt = """
You are a smart data assistant capable of reading multiple CSV files.
- You have access to 4 different datasets: SaaS Docs, Credit Card Terms, Hospital Policy, and Ecommerce FAQs.
- When asked a question, determine which DataFrame is most relevant.
- Do NOT answer from general knowledge.
- Answer in plain English.
"""

try:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0, openai_api_key=api_key)

    agent = create_pandas_dataframe_agent(
        llm,
        dataframes,
        verbose=True,
        agent_type="openai-functions",
        allow_dangerous_code=True
    )

    print("\nAI Agent is ready! You can ask questions across ALL files.")
    print("Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'")

except Exception as e:
    print(f"Error initializing agent: {e}")
    sys.exit()

In [ ]:
# ==========================================
#  PART 3: CHAT LOOP
# ==========================================
print("\nType 'exit' or 'quit' to stop conversation.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    if not user_input:
        continue

    final_query = system_prompt + "\n\nQuestion: " + user_input
    print("AI is thinking...")

    try:
        response = agent.invoke({"input": final_query})

        print(f"AI: {response['output']}\n" + "-"*30)
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
!pip install streamlit

### 1. Create your Streamlit application (app.py)

In [ ]:
%%writefile app.py

import os

import pandas as pd
import streamlit as st
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

st.set_page_config(page_title="Smart CSV Agent", page_icon="\U0001F4CA", layout="wide")

st.title("\U0001F4CA Smart CSV Agent")
st.write("Upload one or more CSV files and ask questions about their content in plain English.")

with st.sidebar:
    st.header("Configuration")
    api_key = st.text_input("OpenAI API Key", value=os.getenv("OPENAI_API_KEY", ""), type="password")
    model_name = st.selectbox("Model", ["gpt-4o-mini", "gpt-4o"], index=0)
    if st.button("Clear chat"):
        st.session_state.messages = []
        st.rerun()

uploaded_files = st.file_uploader("Choose CSV files", accept_multiple_files=True, type=['csv'])

dataframes = []
loaded_names = []

if uploaded_files:
    st.subheader("Uploaded Files")
    for uploaded_file in uploaded_files:
        try:
            df = pd.read_csv(uploaded_file)
            dataframes.append(df)
            loaded_names.append(uploaded_file.name)
            st.success(f"Loaded '{uploaded_file.name}' ({len(df)} rows)")
        except Exception as e:
            st.error(f"Error loading {uploaded_file.name}: {e}")

if "messages" not in st.session_state:
    st.session_state.messages = []

if not uploaded_files:
    st.info("Upload at least one CSV file to get started.")
elif not api_key:
    st.warning("Enter your OpenAI API key in the sidebar to activate the agent.")
elif dataframes:
    system_prompt = """
You are a smart data assistant capable of reading multiple CSV files.
- You have access to the following datasets: {loaded_files}.
- When asked a question, determine which dataset is most relevant.
- Do NOT answer from general knowledge.
- Answer in plain English.
""".format(loaded_files=", ".join(loaded_names))

    try:
        llm = ChatOpenAI(model=model_name, temperature=0.0, openai_api_key=api_key)
        agent = create_pandas_dataframe_agent(
            llm,
            dataframes,
            verbose=True,
            agent_type="openai-functions",
            allow_dangerous_code=True
        )
    except Exception as e:
        st.error(f"Error initializing agent: {e}")
        agent = None

    if agent is not None:
        for message in st.session_state.messages:
            with st.chat_message(message["role"]):
                st.markdown(message["content"])

        if user_input := st.chat_input("Ask a question about your data..."):
            st.session_state.messages.append({"role": "user", "content": user_input})
            with st.chat_message("user"):
                st.markdown(user_input)

            with st.chat_message("assistant"):
                with st.spinner("Thinking..."):
                    try:
                        final_query = system_prompt + "\n\nQuestion: " + user_input
                        response = agent.invoke({"input": final_query})
                        answer = response["output"]
                    except Exception as e:
                        answer = f"An error occurred: {e}"
                    st.markdown(answer)
            st.session_state.messages.append({"role": "assistant", "content": answer})


### 2. Run your Streamlit application

In [ ]:
import subprocess

# Run the Streamlit app in the background
subprocess.Popen(['streamlit', 'run', 'app.py'])


The Streamlit app is now running in the background. To access it, you'll need to use `ngrok` to create a public URL.

### 4. Deploy to Streamlit Community Cloud (Optional)

If you want to deploy your app for a wider audience, you can use Streamlit Community Cloud. Here's how:

1.  **Save your app:** Make sure your `app.py` file (or whatever you named your Streamlit script) is saved.
2.  **Create a GitHub Repository:** Push your `app.py` file and any other necessary files (like a `requirements.txt` listing all your Python dependencies) to a public GitHub repository.
3.  **Go to Streamlit Community Cloud:** Visit [share.streamlit.io](https://share.streamlit.io/).
4.  **Deploy your app:** Click on 'New app', connect your GitHub account, select your repository, the main branch, and the path to your Streamlit app file (e.g., `app.py`).
5.  **Click 'Deploy!':** Streamlit will automatically build and deploy your application. Once deployed, you'll get a public URL for your app.